In [0]:
# %sql
# drop schema pysparkdbt.gold cascade

In [0]:
df = spark.read.format('csv')\
.option('header', True)\
.option('inferSchema', True)\
  .load('/Volumes/pysparkdbt/source/source_data/customers/customers.csv')
display(df.limit(5))

In [0]:
schema_customer = df.schema
schema_customer

### Spark Streaming

In [0]:
# df = spark.readStream.format("csv")\
# .option('header', True)\
# .schema(schema_customer)\
#   .load('/Volumes/pysparkdbt/source/source_data/customers')
  
# # checkpointLocation
# df.writeStream.format("delta")\
# .outputMode("append")\
# .option("checkpointLocation", "/Volumes/pysparkdbt/source/checkpoint/customers")\
#     .trigger(once=True)\
# .toTable("pysparkdbt.bronze.customers")  # this project put table in source layer to manage

In [ ]:
# %sql
# DROP TABLE IF EXISTS pysparkdbt.bronze.customers

In [ ]:
# # delete the checkpoint folder if you want to re-run the ingestion
# dbutils.fs.rm(f"/Volumes/pysparkdbt/source/checkpoint/customers", recurse=True)

### dynamic solution: checkpoint volume bronze schema

In [0]:
entities = ['customers', 'payments', 'locations','trips','vehicles','drivers']

In [0]:
for entity in entities:

    df_batch = spark.read.format('csv')\
    .option('header', True)\
    .option('inferSchema', True)\
    .load(f"/Volumes/pysparkdbt/source/source_data/{entity}/")

    schema_entity = df_batch.schema
  
    df = spark.readStream.format("csv")\
        .option('header', True)\
        .schema(schema_entity)\
        .load(f"/Volumes/pysparkdbt/source/source_data/{entity}")

    df.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", f"/Volumes/pysparkdbt/source/checkpoint/{entity}")\
            .trigger(once=True)\
    .toTable(f"pysparkdbt.source.{entity}")  


In [ ]:
# %sql
# DROP TABLE IF EXISTS pysparkdbt.source.customers;
# DROP TABLE IF EXISTS pysparkdbt.source.payments;
# DROP TABLE IF EXISTS pysparkdbt.source.locations;
# DROP TABLE IF EXISTS pysparkdbt.source.trips;
# DROP TABLE IF EXISTS pysparkdbt.source.vehicles;
# DROP TABLE IF EXISTS pysparkdbt.source.drivers;

In [0]:
%sql
select * from pysparkdbt.source.customers 
order by 1
limit 3

In [ ]:
%sql
select 'customers' as table_name, count(*) as row_count from pysparkdbt.source.customers
union all
select 'payments', count(*) from pysparkdbt.source.payments
union all
select 'locations', count(*) from pysparkdbt.source.locations
union all
select 'trips', count(*) from pysparkdbt.source.trips
union all
select 'vehicles', count(*) from pysparkdbt.source.vehicles
union all
select 'drivers', count(*) from pysparkdbt.source.drivers;

In [0]:
%sql
select * from pysparkdbt.source.drivers 
limit 3

In [0]:
%sql
select * from pysparkdbt.source.locations 
limit 3

In [0]:
%sql
select * from pysparkdbt.source.payments 
limit 3

In [0]:
%sql
select * from pysparkdbt.source.trips 
limit 3

In [0]:
%sql
select * from pysparkdbt.source.vehicles 
limit 3

### test duplicate data

In [ ]:
%sql
select 'customers' as table_name, cast(customer_id as string) as pk_val, count(*) as cnt 
from pysparkdbt.source.customers group by customer_id having count(*) > 1
union all
select 'payments', cast(payment_id as string), count(*) 
from pysparkdbt.source.payments group by payment_id having count(*) > 1
union all
select 'locations', cast(location_id as string), count(*) 
from pysparkdbt.source.locations group by location_id having count(*) > 1
union all
select 'trips', cast(trip_id as string), count(*) 
from pysparkdbt.source.trips group by trip_id having count(*) > 1
union all
select 'vehicles', cast(vehicle_id as string), count(*) 
from pysparkdbt.source.vehicles group by vehicle_id having count(*) > 1
union all
select 'drivers', cast(driver_id as string), count(*) 
from pysparkdbt.source.drivers group by driver_id having count(*) > 1;

### Bronze layer

In [0]:
%sql
select * from pysparkdbt.bronze.stg_customers
limit 3

### Silver layer

In [0]:
%sql
select * from pysparkdbt.silver.silver_customers
limit 5

In [0]:
%sql
select * from pysparkdbt.silver.silver_drivers
limit 5

In [0]:
%sql
select * from pysparkdbt.silver.silver_payments
limit 3

In [0]:
%sql
select * from pysparkdbt.silver.silver_trips 
limit 5

In [ ]:
%sql
select * from pysparkdbt.silver.silver_vehicles
limit 5

In [ ]:
%sql
select * from pysparkdbt.silver.silver_locations
limit 5

In [ ]:
%sql
select 'customers' as table_name, count(*) as row_count from pysparkdbt.silver.silver_customers
union all
select 'payments', count(*) from pysparkdbt.silver.silver_payments
union all
select 'locations', count(*) from pysparkdbt.silver.silver_locations
union all
select 'trips', count(*) from pysparkdbt.silver.silver_trips
union all
select 'vehicles', count(*) from pysparkdbt.silver.silver_vehicles
union all
select 'drivers', count(*) from pysparkdbt.silver.silver_drivers;

In [0]:
%sql
describe table pysparkdbt.source.trips 

In [0]:
%sql
describe table pysparkdbt.source.payments 

In [ ]:
%sql
select * from pysparkdbt.gold.gold_daily_driver_performance